In [39]:
from langgraph.graph import StateGraph,START,END
import cohere
from typing import TypedDict
from dotenv import load_dotenv
import os

In [40]:
load_dotenv()  # Load environment variables from .env file

True

In [41]:
model= cohere.ClientV2(os.getenv("COHERE_API_KEY"))  # Initialize the Cohere client with your API key

In [42]:
print(model.models.list())

models=[GetModelResponse(name='c4ai-aya-expanse-32b', is_deprecated=None, endpoints=['generate', 'chat'], finetuned=False, context_length=128000, tokenizer_url='https://storage.googleapis.com/cohere-public/tokenizers/c4ai-aya-expanse-32b.json', default_endpoints=[], features=None, sampling_defaults=None), GetModelResponse(name='c4ai-aya-vision-32b', is_deprecated=None, endpoints=['chat'], finetuned=False, context_length=16384, tokenizer_url='https://storage.googleapis.com/cohere-public/tokenizers/c4ai-aya-vision-32b.json', default_endpoints=[], features=['logprobs', 'vision', 'citations'], sampling_defaults=None), GetModelResponse(name='cohere-transcribe-03-2026', is_deprecated=None, endpoints=['transcriptions'], finetuned=False, context_length=32768, tokenizer_url='https://storage.googleapis.com/cohere-public/tokenizers/cohere-transcribe-03-2026.json', default_endpoints=[], features=None, sampling_defaults=None), GetModelResponse(name='command-a-03-2025', is_deprecated=None, endpoints

In [43]:
class LLMState(TypedDict):
    prompt: str
    response: str

In [44]:
model_name="command-a-plus-05-2026"

In [45]:
def LLM_QA(state: LLMState) -> LLMState:
    question = state["prompt"]
    response = model.chat(model=model_name, messages=[{"role": "user", "content": question}])
    state["response"] = next(item.text for item in response.message.content if item.type == "text")
    return state


In [46]:
graph=StateGraph(LLMState)
graph.add_node("LLM_QA",LLM_QA)
graph.add_edge(START,"LLM_QA")
graph.add_edge("LLM_QA",END)
workflow=graph.compile()

In [47]:
initial_state={"prompt": "How far is moon from the earth?"}
final_state=workflow.invoke(initial_state)
print(final_state["response"])


The Moon’s distance from Earth isn’t a single fixed number because it varies as the Moon orbits our planet.  

**Average distance**  
- About **384,400 km** (≈ 238,900 mi) from Earth’s center.  

**Closest approach (perigee)**  
- Roughly **363,300 km** (≈ 225,700 mi) when the Moon is at its nearest point in its elliptical orbit.  

**Farthest point (apogee)**  
- About **405,500 km** (≈ 252,000 mi) when the Moon is at its farthest point.  

These figures refer to the straight‑line distance between the centers of Earth and the Moon. If you measure from the surface of Earth, subtract Earth’s radius (~6,371 km or 3,959 mi) to get the distance from the ground to the Moon’s surface.
